# Automatic Deep Research 

Welcome to this new practice lab! By now you should have a clearer view of the elements that compose a multi-agent system. In this lab you will get to put it into action by creating your first crew.

**What you'll learn:**
- How to define agents with specific roles and expertise
- How to provide agents with tools to perform their tasks
- How to create your own tasks that agents will execute
- How to assemble agents and tasks into a Crew, all using CrewAI

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an automatic deep research solution that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter. 

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

If you are stuck, or simply want to copy a solution into your notebook so that you can execute it, you can find all solution code inside the [Solution](Solution) folder.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Understanding the problem](#1)
- [2. Set up your notebook](#2)
- [3. Define the Agents](#3)
  - [3.1. Create tool instances](#3-1)
  - [3.2. Define the Research Planner agent](#3-2)
  - [3.3. Define the remaining agents](#3-3)
- [4. Create the Tasks](#4)
  - [4.1. Define the Create research plan task](#4-1)
  - [4.2. Define the remaining tasks](#4-2)
- [5. Define the Crew and get the results](#5)

<a id="1"></a>

## 1. Understanding the problem
In this lab, you will focus on building a custom deep research crew. This Crew will be in charge of creating a research plan based on the user's input, and executing it, while reviewing and checking the facts. Finally, with the gathered information a report needs to be generated.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task? 

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<img src="../images/lab2-agents-tasks-diagram.PNG">

<a id="2"></a>

## 2. Set up your notebook

Before you start coding, run the next two cells to import all necessary modules and configure the environment variables. 

In [17]:
from crewai import Agent, Task, Crew
import os
from datetime import date

TODAY = date.today().isoformat()

In [11]:
from crewai import Agent, Task, Crew
import os
from datetime import date
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

# Today's date is injected into the tasks so the agents always know
# what "recent" / "latest" / "this week" actually means when they search.

print("Model:", os.environ["MODEL"])
print("Today's date (used for recency grounding):", TODAY)

Model: gpt-4o-mini
Today's date (used for recency grounding): 2026-08-15


In [23]:
!pip install "crewai[google-genai]" -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import google.genai
print("Google GenAI installed")

Google GenAI installed


In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""

In [20]:
from crewai import Agent, LLM

llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0
)

<a id="3"></a>

## 3. Define the Agents

Based on the diagram, you should have four agents:
- **Research Planner**: its goal is to analyze queries and break them down into smaller, specific research topics.
- **Internet Researcher**: its job is to perform research tasks.
- **Fact checker**: its goal is to review information for fact accuracy to avoid misinformation. 
- **Report Writer**: is in charge of writing reports, based on gathered information.

<a id="3-1"></a>

### 3.1. Create tool instances
As you can see in the diagram, you will be providing the **Internet Researcher Agent** with tools, so that it can better do their job. In particular, you will give this agent access to search the internet and scrape information from the retrieved webpages. 

There are different tools inside CrewAI you can use to search the web, in this lab you will use the [**EXA Search Web Loader**](https://docs.crewai.com/en/tools/search-research/exasearchtool#exa-search-web-loader) tool, which is designed to perform a semantic search for a specified query from a text’s content across the internet. It utilizes the [exa.ai](https://exa.ai/) API to fetch and display the most relevant search results based on the query provided by the user. exa.ai enhances semantic search by capturing richer contextual relationships between concepts, allowing for more precise information retrieval than conventional embedding approaches.

For webscraping, you will use the [**Scrape Website**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool) tool, which is designed to extract and read the content of a specified website.

In the next cell you will define instances of these tools, so you can later assign them to the agents.

In [21]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from utils import get_exa_api_key

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

# Create the EXASearchTool instance (web + news semantic search)
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
# Create the ScrapeWebsiteTool instance (full-page content extraction)
scrape_website_tool = ScrapeWebsiteTool()

<a id="3-2"></a>

### 3.2. Define the Research Planner agent

In the cell below, you will see how you can create the first agent. This time, all the parameters are set up for you. Here is a quick recap of what each of the parameters represent:

- `Role`: If this was a person doing the job, what title would they have?
- `Goal`: What is the goal this agent in particular is trying to accomplish? Make sure to write concrete goal
- `Background`: it should be something the highlights the skills of the agent relevant to its role. Make sure to use keywords that will actually help your agent get better results.

In the labs, we have added two parameters not shown in the demo videos: `max_rpm`, and `max_iter`. `max_rpm` sets the maximum requests per minute to avoid rate limits, while `max_iter` limits the maximum iterations before the agent must provide its best answer. Setting these two parameters helps make the agents run a little faster, so the lab doesn't take as long to complete. 

In [22]:
query_analyzer = Agent(
    role="Fact-Check Query Analyzer",
    goal=(
        "Break down the user's query or claim into specific, checkable sub-claims, "
        "identify the exact entities, events, and dates involved, and determine the "
        "correct time window to search (a specific date/period if the user gave one, "
        "otherwise the most recent available information as of {current_date})."
    ),
    backstory=(
        "You are a fact-checking desk editor, similar to those at outlets like TN Fact Check / "
        "TN IID, who specializes in turning vague or loaded user queries into precise, verifiable "
        "research questions. You are extremely careful about time: you always distinguish between "
        "'what happened historically' and 'what is happening now', and you flag explicitly when a "
        "query needs today's or this week's news rather than background information. You never let "
        "an ambiguous claim go unresolved -- you always specify exactly what needs to be verified "
        "and by when the underlying information should be dated."
    ),
    verbose=True,
    max_rpm=150,
    max_iter=15,
    llm=llm
)

<a id="3-3"></a>

### 3.3. Define the remaining agents

Now you can define the three remaining agents. The `role` and `goal` parameters are already filled in for you; use your own creativity to fill in the `backstory`.  

Do not forget to assign the tools to the **Internet Researcher** and **Fact Checker** agents. You can do this by setting the `tools` argument.

In [23]:
news_researcher = Agent(
    role="Real-Time News Researcher",
    goal=(
        "Retrieve the most recent, relevant information available for each research "
        "topic -- prioritizing breaking news, official statements, recent actions, and "
        "reports published within the requested (or most recent possible) time window -- "
        "and record the exact publish date and source URL for everything found."
    ),
    backstory=(
        "You are an investigative wire-service researcher who specializes in real-time "
        "news retrieval. You know that yesterday's article is more useful than last year's, "
        "so you always search with recency-biased terms ('latest', 'today', 'this week', "
        "specific months/years) and sort mentally for the newest credible coverage first. "
        "You cross-reference multiple outlets (news sites, official government/organization "
        "pages, press releases) rather than relying on a single source, and you scrape pages "
        "when a snippet is not enough to confirm a date, figure, or direct quote."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    verbose=True,
    max_rpm=150,
    max_iter=15,
    llm=llm
)

fact_verifier = Agent(
    role="Fact Verification Specialist",
    goal=(
        "Verify every claim and figure gathered by the researcher against at least one "
        "independent, credible, and recent source; flag information that is outdated, "
        "unsupported, contradictory, or based on a single unverified source; and rate "
        "the recency and reliability of each source used."
    ),
    backstory=(
        "You are a meticulous fact-checking specialist in the mold of TN Fact Check / TN IID "
        "verification desks. You never accept a claim at face value -- you actively re-search "
        "and cross-check it against independent sources, paying special attention to publish "
        "dates so that outdated information is never presented as current. You clearly label "
        "each claim as Verified, Partially Verified, Unverified, Outdated, or False, and you "
        "explain exactly why."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    verbose=True,
    max_rpm=150,
    max_iter=15,
    llm=llm
)

factcheck_report_writer = Agent(
    role="Fact-Check Report Writer",
    goal=(
        "Write a clear, well-structured fact-check report that gives the user a direct "
        "verdict on their query, backed by the verified and up-to-date evidence, with "
        "full source citations including publish dates."
    ),
    backstory=(
        "You are a professional fact-check report writer who transforms verification "
        "findings into a concise, publication-ready verdict -- similar to how outlets "
        "like TN Fact Check / TN IID present their conclusions. You lead with a clear "
        "rating (True / False / Misleading / Unverified / Needs More Context), summarize "
        "the evidence in plain language, explicitly note how recent the information is, "
        "and list every source with a link and publish date."
    ),
    verbose=True,
    max_rpm=150,
    max_iter=15,
    llm=llm
)

<a id="4"></a>

## 4. Create the Tasks

Now that you have set up the agents, it is time to define the tasks. If you go back to the diagram, you will see you need four tasks:

- **Create research plan**: Based on the user's query, break it down into specific topics and key questions, and create a focused research plan.
    - Output: A research plan with main research topics to investigate, key questions for each topic, and success criteria for the research.

- **Gather research data**: Using the research plan, collect information on all identified topics. Cite all sources used.
    - Output: Comprehensive research data including: information for each research topic, and citations used along with source credibility notes.

- **Verify information quality**: Review all collected research. Identify any conflicting information, potential misinformation, or gaps that need addressing.
    - Output: A report with the all the collected data, and its review. It should include consistency check results and source reliability ratings

- **Write final report**: Create a comprehensive report that answers the original query using all verified research data. Structure it with clear sections, include citations, and provide actionable insights.
    - Output: The final research report. In addition to the full answer, it should have an executive summary, and complete source citations.


For each `Task` you need to define the following parameters:
- `description`: A thorough description of the task. You can even break it down into different items.
- `expected_output`: what should the output return. Be specific, specially if you want any structure in your result, like a dictionary with specific keys.
- `agent`: who is performing the task? You need to match the task to one of the agents you already defined

In the description you will need to pass the inputs to the tasks. In this lab, you will only have as input the user's query, which will be saved as `user_query`:


<a id="4-1"></a>

### 4.1. Define the Create research plan task

In the cell below, you will see how you can create the first task. This time, all the parameters are set up for you. Notice how the context variables are passed the the description between curly brackets. 

In [24]:
analyze_query_task = Task(
    description=(
        "Analyze the user's query and break it down into specific, checkable sub-claims "
        "and key questions. Identify all entities, events, organizations, and dates involved. "
        "Determine the correct time window for research: if the user specified a date or period, "
        "use exactly that; otherwise, target the most recent information available as of "
        "{current_date}. Produce a focused research plan with concrete, recency-biased search "
        "queries (e.g. including terms like 'latest', 'today', 'this week', or the relevant year/month) "
        "for the researcher to use.\n\n"
        "The user's query is: {user_query}\n"
        "The user-specified time period (if any) is: {time_period}\n"
        "Today's date is: {current_date}"
    ),
    expected_output=(
        "A research plan listing: (1) the specific sub-claims/questions to verify, "
        "(2) the exact entities/events/dates involved, (3) the determined time window "
        "to search within, and (4) a set of concrete, recency-biased search queries "
        "for each sub-claim."
    ),
    agent=query_analyzer,
)

<a id="4-2"></a>

### 4.2. Define the remaining tasks

Now define the three remaining tasks. The `description` is already filled in for you, you will need to define the `expected_output` and `agent` for each of the Tasks.

In [25]:
# define the retrieve recent information task
retrieve_recent_info_task = Task(
    description=(
        "Using the research plan, search the web and news sources for the most recent, "
        "relevant information on every identified sub-claim and topic. Prioritize sources "
        "published within the determined time window -- use recency-biased search terms "
        "and, when the user gave no specific period, actively seek out the latest available "
        "news, actions, or developments rather than older background material. For every "
        "piece of information gathered, record the source URL and its publish date, and "
        "scrape the page when needed to confirm exact dates, figures, or quotes."
    ),
    expected_output=(
        "A detailed collection of research findings covering every sub-claim, each entry "
        "including the finding itself, the source URL, and the source's publish date, "
        "clearly separating the most recent findings from any older/background context."
    ),
    agent=news_researcher
)

# define the verify facts task
verify_facts_task = Task(
    description=(
        "Review all gathered research. For each claim or finding, verify it against at "
        "least one independent, credible source. Identify any conflicting information, "
        "outdated claims (information superseded by more recent developments), potential "
        "misinformation, or gaps that still need addressing. Explicitly check whether each "
        "source's publish date falls within the required time window; if a source is stale, "
        "search for a more recent update and note whether the situation has changed."
    ),
    expected_output=(
        "A fact-verification summary that, for each sub-claim, states a status "
        "(Verified / Partially Verified / Unverified / Outdated / False), the supporting "
        "evidence, any conflicting claims found, a note on source recency and reliability, "
        "and any recommended corrections or additional research needed."
    ),
    agent=fact_verifier
)

# define the write final fact-check report task
write_factcheck_report_task = Task(
    description=(
        "Create a final fact-check report that directly answers the user's original query "
        "using only the verified, up-to-date research. Lead with a clear overall verdict "
        "(True / False / Misleading / Unverified / Needs More Context). Summarize the "
        "supporting evidence in plain language, explicitly state how recent the underlying "
        "information is (mention specific dates), and list every source used with its link "
        "and publish date."
    ),
    expected_output=(
        "A comprehensive, clearly structured fact-check report containing: an overall verdict, "
        "an executive summary, a detailed evidence breakdown per sub-claim, an explicit note on "
        "the recency of the information used, and a complete list of source citations with "
        "publish dates."
    ),
    agent=factcheck_report_writer
)

<a id="5"></a>

## 5. Define the Crew and get the results

Once the agents and tasks have been defined, you are ready to create the crew. In order to so, you will need to set the following arguments:
- `agents`: list of agents in the crew
- `tasks`: list of tasks in the crew. The tasks should be listed in the order they should be executed

In the next cell, fill in the agents and tasks for the crew.

In [26]:
# create the crew with the defined agents and tasks
crew = Crew(
    agents=[query_analyzer, news_researcher, fact_verifier, factcheck_report_writer],
    tasks=[analyze_query_task, retrieve_recent_info_task, verify_facts_task, write_factcheck_report_task]
)

Before running the crew, you need to define the query, which will be used as input for the tasks.

In [30]:
# Write the query/claim you want fact-checked
user_query = "MK stalin won in recounting in kolathur"

# Optional: pin the fact-check to a specific date/period.
# Leave as "not specified" to default to the most recent available information.
time_period = "August 2026"

Now you are only left with kickstarting the crew to get the results. Since you set `verbose=True` in the agents, you should monitor all the process.

In [31]:
result = crew.kickoff(
    inputs={
        "user_query": user_query,
        "time_period": time_period,
        "current_date": TODAY,
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Query Analyzer                                                                               │
│                                                                                                                 │
│  Task: Analyze the user's query and break it down into specific, checkable sub-claims and key questions.        │
│  Identify all entities, events, organizations, and dates involved. Determine the correct time window for        │
│  research: if the user specified a date or period, use exactly that; otherwise, target the most recent          │
│  information available as of 2026-08-15. Produce a focused research plan with concrete, recency-biased search   │
│  queries (e.g. including terms like 'latest', 'today', 'this week', or the relevant year/month) for the         │
│  researcher to use.                                                                                             │
│                                                                                                                 │
│  The user's query is: MK stalin won in recounting in kolathur                                                   │
│  The user-specified time period (if any) is: August 2026                                                        │
│  Today's date is: 2026-08-15                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Query Analyzer                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan for "MK stalin won in recounting in kolathur"**                                                │
│                                                                                                                 │
│  **1. Specific Sub-claims/Questions to Verify:**                                                                │
│                                                                                                                 │
│  *   Is there an election (general assembly election or by-election) scheduled or ongoing in the Kolathur       │
│  constituency, Tamil Nadu, in August 2026?                                                                      │
│  *   If an election is confirmed for August 2026, is M.K. Stalin a declared candidate for the Kolathur          │
│  constituency in that election?                                                                                 │
│  *   If M.K. Stalin is a candidate and an election has occurred, has there been a request for or an actual      │
│  recounting of votes in the Kolathur constituency in August 2026?                                               │
│  *   If a recounting occurred, what were the official results, and did M.K. Stalin win the election in          │
│  Kolathur after this recounting in August 2026?                                                                 │
│                                                                                                                 │
│  **2. Exact Entities/Events/Dates Involved:**                                                                   │
│                                                                                                                 │
│  *   **Entities:** M.K. Stalin, Kolathur constituency (Tamil Nadu, India), Election Commission of India (or     │
│  Tamil Nadu State Election Commission), Dravida Munnetra Kazhagam (DMK) party.                                  │
│  *   **Events:** State Assembly Election (or by-election), Candidacy Declaration, Vote Recounting, Election     │
│  Results Announcement, Election Victory.                                                                        │
│  *   **Dates:** Specifically August 2026. The information should be current as of 2026-08-15.                   │
│                                                                                                                 │
│  **3. Determined Time Window to Search Within:**                                                                │
│                                                                                                                 │
│  *   August 2026 (with a focus on news and official announcements from 2026-08-01 to 2026-08-15, and any        │
│  preceding announcements for events occurring in August 2026).                                                  │
│                                                                                                                 │
│  **4. Concrete, Recency-Biased Search Queries for Each Sub-claim:**                                             │
│                                                                                                                 │
│  *   **For Sub-claim 1 (Election in Kolathur, August 2026):**                                                   │
│      *   `"Kolathur election August 2026"`             

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Real-Time News Researcher                                                                               │
│                                                                                                                 │
│  Task: Using the research plan, search the web and news sources for the most recent, relevant information on    │
│  every identified sub-claim and topic. Prioritize sources published within the determined time window -- use    │
│  recency-biased search terms and, when the user gave no specific period, actively seek out the latest           │
│  available news, actions, or developments rather than older background material. For every piece of             │
│  information gathered, record the source URL and its publish date, and scrape the page when needed to confirm   │
│  exact dates, figures, or quotes.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Real-Time News Researcher                                                                               │
│                                                                                                                 │
│  Thought: Action: EXASearchTool                                                                                 │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "\"Kolathur election August 2026\"",                                                         │
│    "start_published_date": "2026-08-01",                                                                        │
│    "end_published_date": "2026-08-15"                                                                           │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: Kolathur Election Results 2026 Declared: M.K Stalin looses after 15 years; TVK sweeps DMK's fortress |  │
│  News | Zee News                                                                                                │
│  URL:                                                                                                           │
│  https://zeenews.india.com/india/live-updates/kolathur-election-results-2026-live-updates-mk-stalin-dmk-vs-aia  │
│  dmk-aadirajaram-vs-tvk-vs-babu-3043281.html                                                                    │
│  ID:                                                                                                            │
│  https://zeenews.india.com/india/live-updates/kolathur-election-results-2026-live-updates-mk-stalin-dmk-vs-aia  │
│  dmk-aadirajaram-vs-tvk-vs-babu-3043281.html                                                                    │
│  Score: None                                                                                                    │
│  Published Date: 2026-08-12T00:00:00.000Z                                                                       │
│  Author: None                                                                                                   │
│  Image: https://english.cdn.zeenews.com/sites/default/files/2026/05/04/1938964-kolathur.jpg.jpeg                │
│  Favicon: None                                                                                                  │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text: Kolathur Election Results 2026 Declared: M.K Stalin looses after 15 years; TVK sweeps DMK's fortress |   │
│  News | Zee News                                                                                                │
│                                                                                                                 │
│  # Kolathur Election Results 2026 Declared: M.K Stalin looses after 15 years; TVK sweeps DMK's fortress         │
│                                                                                                                 │
│  ## Kolathur Election Results 2026 Declared: One of Tamil Nadu’s high-stakes seats was locked in a triangular   │
│  battle between DMK, AIADMK and TVK. CM M.K. MK Stalin looses his stronghold after 15 years, TVK’s VS Babu      │
│  sweeps by 8795 vote margin. Stalin won the seat by 41% margin in 2021. DMK looses its bastion after 15 years.  │
│                                                                                                                 │
│  Written By Zee Media Bureau Edited By Anjali Singh                                                             │
│                                                                                                                 │
│  Published: May 03, 2026, 08:17 PM IST| Updated: May 04, 2026, 06:07 PM IST                                     │
│                                                                                                                 │
│  04 May 2026 16:40 IST (IST)                                                                                    │
│                                                                                                                 │
│  ### Kolathur Election Results 2026 Live: MK Stalin looses his den after 15 years; TVKs VS Babu marks historic  │
│  winKolathur Election Results 2026: DMK looses its bastion after 15 years; TVK scripts history. MK Stalin       │
│  trails with massive margin of 8284 vot               

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Verification Specialist                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│  Thought:The search results confirm what the provided research indicated:                                       │
│  *   The "Kolathur Election Results 2026 Declared" in May 2026 (Zee News, published May 3, 2026, updated May    │
│  4, 2026, and also a search result with a published date of 2026-08-12, which seems to be a re-indexing or      │
│  update of the original May article).                                                                           │
│  *   The August 2026 events are consistently referred to as "EVM Checking and Verification process" or "EVM     │
│  verification," not an election or recounting, as explicitly stated by election officials (The Hindu, August    │
│  1, 2026).                                                                                                      │
│  *   The Wikipedia page for "17th Tamil Nadu Assembly" states it was constituted after the "April 2026 State    │
│  Assembly elections" and the term began "11 May 2026".                                                          │
│                                                                                                                 │
│  Therefore, Sub-claim 1 is False. There was no election in Kolathur in August 2026. The election was in         │
│  April/May 2026.                                                                                                │
│                                                                                                                 │
│  Now, let's move to Sub-claim 2.                                                                                │
│                                                                                                                 │
│  **Sub-claim 2: If an election is confirmed for August 2026, is M.K. Stalin a declared candidate for the        │
│  Kolathur constituency in that election?**                                                                      │
│                                                                                                                 │
│  *   Since Sub-claim 1 is False (no election in August 2026), this sub-claim is also False. M.K. Stalin was a   │
│  candidate in the *May 2026* election.                                                                          │
│                                                                                                                 │
│  Let's move to Sub-claim 3.                                                                                     │
│                                                                                                                 │
│  **Sub-claim 3: If M.K. Stalin is a candidate and an election has occurred, has there been a request for or an  │
│  actual recounting of votes in the Kolathur constituency in August 2026?**                                      │
│                                                                                                                 │
│  *   The research and search results consistently refer to an "EVM verification process," not a "recounting."   │
│  *   **Source:** "No evidence of EVM tampering, says poll official on fourth day," The Hindu, Published:        │
│  2026-08-01, URL:                                      

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Report Writer                                                                                │
│                                                                                                                 │
│  Task: Create a final fact-check report that directly answers the user's original query using only the          │
│  verified, up-to-date research. Lead with a clear overall verdict (True / False / Misleading / Unverified /     │
│  Needs More Context). Summarize the supporting evidence in plain language, explicitly state how recent the      │
│  underlying information is (mention specific dates), and list every source used with its link and publish       │
│  date.                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Report Writer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Fact-Check Report: "MK stalin won in recounting in kolathur"**                                               │
│                                                                                                                 │
│  **Overall Verdict: False**                                                                                     │
│                                                                                                                 │
│  **Executive Summary:**                                                                                         │
│  The claim that M.K. Stalin won in a recounting in Kolathur in August 2026 is **False**. The Tamil Nadu         │
│  Assembly election for the Kolathur constituency was held in April 2026, with results declared on May 3-4,      │
│  2026. In that election, M.K. Stalin lost the Kolathur seat to V.S. Babu of the Tamilaga Vettri Kazhagam (TVK)  │
│  by a margin of 8,795 votes. While an "EVM Checking and Verification process" was conducted in late July and    │
│  early August 2026 following allegations of discrepancies by the DMK, election officials explicitly stated      │
│  that this was not a vote recounting. This verification process concluded with officials confirming no          │
│  evidence of tampering or irregularities that would alter the original election outcome. Therefore, M.K.        │
│  Stalin did not win in any recounting in Kolathur in August 2026.                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Detailed Evidence Breakdown:**                                                                               │
│                                                                                                                 │
│  **Sub-claim 1: Is there an election (general assembly election or by-election) scheduled or ongoing in the     │
│  Kolathur constituency, Tamil Nadu, in August 2026?**                                                           │
│  *   **Status:** False                                                                                          │
│  *   **Supporting Evidence:**                                                                                   │
│      *   The Tamil Nadu Assembly election for the Kolathur constituency, where M.K. Stalin was a candidate,     │
│  concluded with results declared on May 3-4, 2026.                                                              │
│      *   The 17th Tamil Nadu Assembly was constituted after the April 2026 State Assembly elections, with its   │
│  term commencing on May 11, 2026.                                                                               │
│      *   News reports from August 2026 consistently refer to an "EVM Checking and Verification process"         │
│  related to the *previous* election, not a new election or by-election. Chennai District Election Officer G.S.  │
│  Sameeran explicitly clarified that the August events constituted an "EVM Checking and Verification process,"   │
│  not a recounting.                                     

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 7862dc93-a28f-434d-851d-c13d1a0e5ab4                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/7862dc93-a28f-434 │
│ d-851d-c13d1a0e5ab4?access_code=TRACE-133ce7efac                             │
│ 🔑 Access Code: TRACE-133ce7efac                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


From the output of the previous cell check all the outputs for each task. Do they match what you expected? If not, go back and refine the `expected_output`. 

You can also print the final report to see the final result of the crew

In [29]:
from IPython.display import Markdown
Markdown(result.raw) 

**VERDICT: MISLEADING**

**Executive Summary:**
While the Greater Chennai Corporation (GCC), a body under the Tamil Nadu government, initiated a "reassessment drive" in early August 2026 that led to significant property tax increases for approximately 3.5 lakh "under-assessed" properties, this action was met with widespread public and political backlash. Consequently, the GCC announced the **suspension** of these revised property tax assessments on August 13-14, 2026. Therefore, while an action resulting in a hike for many was undertaken, it was not a general, uniform property tax hike across the board, and the revised assessments have since been put on hold.

**Evidence Breakdown:**

1.  **Did the Tamil Nadu government issue any official announcement or notification regarding a hike in house/property tax?**
    *   **Yes, an action was taken by a government body (Greater Chennai Corporation) that resulted in significant tax increases for many properties, though it was termed a "reassessment" rather than a general "hike."**
    *   The Greater Chennai Corporation (GCC) initiated a "reassessment drive" for property tax, issuing revised demand notices to approximately 3.5 lakh property owners in Chennai. The GCC Commissioner, G.S. Sameeran, clarified that this was not a general property tax hike but a correction for properties that were "under-assessed" or "wrongly-assessed" based on discrepancies identified through GIS mapping, satellite data, and other records. This action was implemented through an administrative order.
    *   However, this move was widely perceived as a hike due to the substantial increases in tax bills for affected residents and faced strong condemnation from political parties like the Communist Party of India (CPI), which urged the Tamil Nadu government to withdraw the "anti-democratic property tax hike."

2.  **If an announcement was made, what are the specific details of the proposed hike (e.g., percentage increase, categories of properties affected, effective date, reasons cited)?**
    *   **Nature of the Action:** The GCC undertook a "reassessment drive" to correct existing property tax assessments for properties deemed "under-assessed" or "wrongly-assessed."
    *   **Methodology:** Discrepancies were identified using Geographic Information System (GIS) mapping, satellite data, other government records, and self-declarations submitted by property owners during a 2018 survey.
    *   **Categories of Properties Affected:** Residential and commercial properties where the actual built-up area or usage was not correctly reflected in their existing assessments.
    *   **Percentage Increase (Reported):** While not a uniform percentage hike, residents reported significant increases ranging from 100% to 400%, with some bills increasing five-fold. Specific examples included a tax bill rising from ₹295 to ₹3,255 and another from ₹6,500 to over ₹31,000. The CPI cited a proposed hike ranging from 25% for residential buildings under 600 sq ft to 100% for those above 1800 sq ft, and potentially 100% for commercial properties and vacant plots.
    *   **Effective Date:** The revised rates were being implemented in August 2026.
    *   **Reasons Cited:** The GCC stated its aim was to correct long-standing disparities in taxation, ensure uniform assessment, and address a severe financial crunch, expecting to generate an additional revenue of ₹83 crore to ₹170 crore annually.
    *   **Controversy and Suspension:** The reassessment drew significant criticism for its sudden implementation, lack of prior notice, and failure to consult the elected council. Due to this strong backlash, the Greater Chennai Corporation **suspended the revised property tax assessments** for "under-assessed" properties, effective immediately, on August 13-14, 2026. For taxpayers who had already paid the revised amount, the excess will be adjusted as an advance payment towards their property tax dues for subsequent half-years.

**Recency of Information:**
The information presented is current as of **August 14, 2026**, based on news reports published between August 12, 2026, and August 14, 2026. The most critical update regarding the suspension of the revised assessments was reported on August 13-14, 2026.

**Sources:**

*   **Times of India:** "Facing backlash, GCC suspends property tax hike"
    *   URL: https://timesofindia.indiatimes.com/city/chennai/facing-backlash-gcc-suspends-property-tax-hike/articleshow/133223949.cms
    *   Publish Date: August 13, 2026 (Author: Aug 14, 2026, 01:48 IST)
*   **Live Chennai:** (Details on suspension of revised property tax)
    *   URL: https://www.livechennai.com/detailnews.asp?newsid=82279
    *   Publish Date: August 14, 2026, 03:44:04.000Z
*   **News18:** "CPI urges Tamil Nadu govt to withdraw anti-democratic property tax hike"
    *   URL: https://one.news18.com/english/article/tamil-nadu/cpi-urges-tamil-nadu-govt-to-withdraw-anti-democratic-property-tax-hike-tam-982186793
    *   Publish Date: August 14, 2026, 07:13:52.000Z
*   **News18:** "Chennai Property Tax Doubles: Satellite Calculation Errors, Commissioner Clarifies"
    *   URL: https://one.news18.com/english/article/photogallery/tamil-nadu/chennai-property-tax-doubles-satellite-calculation-errors-commissioner-clarifies-tam-982186234
    *   Publish Date: August 13, 2026, 14:05:02.000Z
*   **The Hindu:** "Corporation’s property tax reassessment drive draws flak for lack of transparency"
    *   URL: https://www.thehindu.com/news/national/tamil-nadu/corporations-property-tax-reassessment-drive-draws-flak-for-lack-of-transparency/article71337856.ece
    *   Publish Date: August 12, 2026 (Updated: August 13, 2026, 12:02 pm IST)
*   **Deccan Chronicle:** "TN GCC eyes under-assessed, wrongly-assessed properties for tax revision"
    *   URL: https://www.deccanchronicle.com/southern-states/tamil-nadu/tn-gcc-eyes-under-assessed-wrongly-assessed-properties-for-tax-revision-1978777
    *   Publish Date: August 13, 2026, 00:00:00.000Z
*   **New Kerala:** "Chennai Corporation clarifies property tax revision amid complaints"
    *   URL: https://www.newkerala.com/news/a/chennai-corporation-clarifies-property-tax-revision-amid-complaints-505.htm
    *   Publish Date: August 13, 2026, 00:00:00.000Z
*   **The News Mill:** "Property tax rates remain unchanged, Greater Chennai Corporation refutes false claims"
    *   URL: https://thenewsmill.com/2026/08/property-tax-rates-remain-unchanged-greater-chennai-corporation-refutes-false-claims/
    *   Publish Date: August 12, 2026, 00:00:00.000Z

You made it to the end of the lab! You can go back and experiment with the goals and backstories of the agents, as well as description and expected outputs of tasks. You can also change the inputs to any research topic you wish. Have fun with it!